<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 13
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Exploratory Data Analysis
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

In [ ]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Example EDA", "local[*]")
su.spark

In [ ]:
!pwd

In [ ]:
!head data/movies/movies_dataset.csv

In [ ]:
# MovieID,Title,Genre,ReleaseYear,ReleaseDate,Country,BudgetUSD,US_BoxOfficeUSD,Global_BoxOfficeUSD,Opening_Day_SalesUSD,One_Week_SalesUSD,IMDbRating,RottenTomatoesScore,NumVotesIMDb,NumVotesRT,Director,LeadActor
movies_columns = [("MovieID", "int"),
                  ("Title", "string"),
                  ("Genre", "string"),
                  ("ReleaseYear", "int"),
                  ("ReleaseDate", "string"),
                  ("Country", "string"),
                  ("BudgetUSD", "float"),
                  ("US_BoxOfficeUSD", "float"),
                  ("Global_BoxOfficeUSD", "float"),
                  ("Opening_Day_SalesUSD", "float"),
                  ("One_Week_SalesUSD", "float"),
                  ("IMDbRating", "float"),
                  ("RottenTomatoesScore", "float"),
                  ("NumVotesIMDb", "int"),
                  ("NumVotesRT", "int"),
                  ("Director", "string"),
                  ("LeadActor", "string")]

movies_schema = SparkUtils.generate_schema(movies_columns)

movies_date_str_df = (su.spark.read 
                .option("header", "true")
                .schema(movies_schema)
                .csv("/opt/spark/work-dir/data/movies/movies_dataset.csv"))
movies_date_str_df = movies_date_str_df.withColumnRenamed("ReleaseDate", "ReleaseDateStr")
movies_date_str_df.printSchema()
movies_date_str_df.show(5)

In [ ]:
null_count_df = SparkUtils.count_nulls(movies_date_str_df)
null_count_df.show()

In [ ]:
import pyspark.sql.functions as F

movies_df = movies_date_str_df.withColumn("ReleaseDate", F.to_date("ReleaseDateStr", "dd-MM-yyyy")).drop("ReleaseDateStr")
movies_df.show(5)

In [ ]:
movies_df.printSchema()

In [ ]:
!pip install plotly

# Line Charts
## Movies produced per year

In [ ]:
sdf = (movies_df
       .groupBy("ReleaseYear")
       .count()
       .orderBy("ReleaseYear"))

fig = sdf.plot.line(x="ReleaseYear", y="count", title="Line: Movies Produced per Year")
fig.show()

## Average global revenue per year

In [ ]:
sdf = (movies_df
       .groupBy("ReleaseYear")
       .agg(F.avg("Global_BoxOfficeUSD").alias("avg_global"))
       .orderBy("ReleaseYear"))

fig = sdf.plot.line(x="ReleaseYear", y="avg_global", title="Line: Avg Global Box Office per Year")
fig.show()

# Area Charts
## Total global revenue per year (area)

In [ ]:
sdf = (movies_df
       .groupBy("ReleaseYear")
       .agg(F.sum("Global_BoxOfficeUSD").alias("sum_global"))
       .orderBy("ReleaseYear"))

fig = sdf.plot.area(x="ReleaseYear", y="sum_global", title="Area: Total Global Box Office per Year")
fig.show()

## Stacked-ish area by Genre (top genres)

In [ ]:
# pick top 5 genres by count
top_genres = [r["Genre"] for r in (movies_df.groupBy("Genre").count()
                                  .orderBy(F.col("count"), ascending=False)
                                  .limit(5).collect())]

print(top_genres)
sdf = (movies_df.filter(F.col("Genre").isin(top_genres))
       .groupBy("ReleaseYear", "Genre")
       .pivot("Genre", top_genres)
       .count()
       .orderBy("ReleaseYear"))

fig = sdf.plot.area(x="ReleaseYear", y=top_genres, title="Area (Stacked): Movie Count per Year by Genre (Top 5)")
fig.show()

# Bar & Column Charts
## Top genres by Genre count (bar)

In [ ]:

sdf = (movies_df.groupBy("Genre")
       .count()
       .orderBy(F.col("count"), ascending=False)
       .limit(10))

fig = sdf.plot.barh(x="count", y="Genre", title="Bar: Top Genres by Movie Count")
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

## Average IMDb rating by country (top 10 by count)

In [ ]:
sdf = (movies_df.groupBy("Country")
       .agg(F.count("*").alias("freq"),
            F.avg("IMDbRating").alias("avg_imdb"))
       .where(F.col("freq") >= 20)
       .orderBy(F.col("freq"), ascending=False)
       .limit(10))

fig = sdf.plot.bar(x="Country", y="avg_imdb", 
                    title="Column: Avg IMDb Rating (countries with >=20 movies, top 10 by volume)")
fig.show()

# Pie & Donut Charts
## Pie: share of top genres (+ Other)

In [ ]:
N = 10
counts = movies_df.groupBy("Genre").count().orderBy(F.col("count"), ascending=False).limit(N)

fig = counts.plot.pie(x="Genre", y="count")
fig.update_layout(title="Share of the top 5 genres")
fig.show()

## Donut: same data

In [ ]:
fig = counts.plot.pie(x="Genre", y="count")
fig.update_traces(hole=0.6)
fig.update_layout(title="Share by genres")
fig.show()

# Histograms

In [ ]:
fig = movies_df.plot.hist(column="IMDbRating", bins=20)
fig.update_layout(xaxis_title="IMDbRating",
                  yaxis_title="Number of movies")
fig.show()

# Box Plots

In [ ]:
fig = movies_df.plot.box(column=["RottenTomatoesScore", "IMDbRating"])
fig.show()

In [ ]:
import plotly.graph_objects as go
 
q = (movies_df.groupBy("genre")
       .agg(F.percentile_approx("IMDbRating",
            [0.05, 0.25, 0.5, 0.75, 0.95]).alias("p"))
       .collect())                        # one small row per genre
 
fig = go.Figure(go.Box(
    x=[r["genre"] for r in q],
    lowerfence=[r["p"][0] for r in q], q1=[r["p"][1] for r in q],
    median=[r["p"][2] for r in q],     q3=[r["p"][3] for r in q],
    upperfence=[r["p"][4] for r in q]))
fig.show()

# Correlation Heatmaps

In [ ]:
import plotly.express as px
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
 
cols = ["IMDbRating", "NumVotesIMDb", "BudgetUSD", "US_BoxOfficeUSD"]
vec = VectorAssembler(inputCols=cols, outputCol="features",
                      handleInvalid="skip").transform(movies_df)
corr = Correlation.corr(vec, "features").head()[0].toArray()
 
fig = px.imshow(corr, x=cols, y=cols, text_auto=".2f",
                zmin=-1, zmax=1, color_continuous_scale="RdBu_r")
fig.show()

## Question 1: Which genres dominate the dataset?

## Question 2: Which countries produce the most successful films?

## Question 3: Which directors produce the highest rated movies after 2010?

## Question 4: Do movies with more Rotten Tomatoes votes get better scores? 

Visualize the relationship between NumVotesRT and RottenTomatoesScore, and say whether popularity and critical approval seem related.

In [ ]:
su.spark.stop()